# Synthetic E-commerce Database (7 Tables): a first look and ten SQL questions

An online outdoor-gear store, from 1 January 2024 to 30 September 2026. Customers in twelve countries buy tents, boots and backpacks through the web shop, the app, a marketplace and a physical store. Seven tables hold the whole order lifecycle: who bought, what was in the basket, how it was paid, when it shipped and what came back.

This notebook opens the SQLite file that ships with the dataset, shows each table, and answers ten questions with SQL. Every query runs as written, so you can change one and re-run it. The same data is also in the CSV files, one per table.

In [1]:
import shutil
import sqlite3
import tempfile
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# On Kaggle the file is under /kaggle/input; elsewhere, put ecommerce.sqlite next to this notebook.
found = sorted(Path("/kaggle/input").rglob("ecommerce.sqlite")) if Path("/kaggle/input").exists() else []
source = found[0] if found else Path("ecommerce.sqlite")

# Work on a copy with an index on every id column: joins become fast and the original stays untouched.
work = Path(tempfile.mkdtemp()) / source.name
shutil.copyfile(source, work)
con = sqlite3.connect(work)
for (table,) in con.execute("SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%'").fetchall():
    for column in [row[1] for row in con.execute(f'PRAGMA table_info("{table}")')]:
        if column.endswith("_id"):
            con.execute(f'CREATE INDEX IF NOT EXISTS "ix_{table}_{column}" ON "{table}" ("{column}")')


def q(sql):
    """Run a query and return the result as a table."""
    return pd.read_sql_query(sql, con)

## The tables

In [2]:
pd.DataFrame(
    [(t, con.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0])
     for (t,) in con.execute("SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%'")],
    columns=["table", "rows"])

,table,rows
0,customers,26000
1,products,1500
2,orders,90000
3,order_items,200000
4,payments,90000
5,shipments,83885
6,returns,8000


### `customers`

In [3]:
q("SELECT * FROM customers LIMIT 5")

,customer_id,first_name,last_name,email,phone,country,city,signup_date,loyalty_tier,marketing_opt_in
0,1,Jill,Ellis,jill.ellis1@example.com,None,France,Marseille,2022-08-06,silver,0
1,2,Darrell,Gilbert,darrell.gilbert2@example.com,+1-873-555-0199,United States,New York,2023-01-02,none,0
2,3,Phillip,Francis,phillip.francis3@example.com,+44 7700 900107,United Kingdom,Leeds,2023-05-05,none,1
3,4,Heather,Reyes,heather.reyes4@example.com,None,France,Paris,2022-05-27,none,0
4,5,Brent,Mayer,brent.mayer5@example.com,+1-514-555-0123,United States,Houston,2023-10-31,none,1


### `products`

In [4]:
q("SELECT * FROM products LIMIT 5")

,product_id,sku,product_name,category,brand,unit_price,unit_cost,weight_kg,is_active,created_at
0,1,OUT-00001,Nomad Down Vest (Sand),Apparel,Stonebrook,161.96,79.21,4.68,1,2022-04-18
1,2,OUT-00002,Ridge Family Dome Tent (Cobalt),Tents,Ironwood Trail,370.93,236.28,3.45,1,2022-02-09
2,3,OUT-00003,Ridge Ultralight Tent (Cobalt),Tents,Larkspur,145.98,101.21,4.91,0,2021-07-31
3,4,OUT-00004,Basecamp Sleeping Pad (Moss),Sleeping Bags,Halden,230.20,132.45,3.94,1,2023-03-27
4,5,OUT-00005,Tundra Camp Lantern (Cobalt),Lighting,Wildhaven,16.81,7.93,4.14,1,2022-04-14


### `orders`

In [5]:
q("SELECT * FROM orders LIMIT 5")

,order_id,customer_id,order_date,status,channel,subtotal,discount_amount,shipping_fee,tax_amount,total_amount,currency
0,1,1,2026-04-18 22:37:25,delivered,app,14.29,0.00,7.73,1.14,23.16,USD
1,2,1,2025-06-09 08:05:40,delivered,marketplace,630.95,0.00,0.00,50.48,681.43,USD
2,3,1,2024-11-29 08:01:39,delivered,app,345.06,62.79,0.00,22.58,304.85,USD
3,4,1,2024-11-19 20:11:54,delivered,web,431.38,0.00,0.00,34.51,465.89,USD
4,5,2,2025-12-28 04:22:31,delivered,web,613.40,0.00,0.00,49.07,662.47,USD


### `order_items`

In [6]:
q("SELECT * FROM order_items LIMIT 5")

,order_item_id,order_id,product_id,quantity,unit_price,line_total
0,1,1,1479,1,14.29,14.29
1,2,2,1201,1,22.48,22.48
2,3,2,587,2,33.35,66.70
3,4,2,219,1,399.09,399.09
4,5,2,1229,1,142.68,142.68


### `payments`

In [7]:
q("SELECT * FROM payments LIMIT 5")

,payment_id,order_id,method,amount,paid_at,status
0,1,30103,card,2433.13,2025-11-13 15:56:38,captured
1,2,58746,paypal,549.37,2024-11-02 18:24:35,captured
2,3,32495,card,164.07,2024-08-27 19:03:04,captured
3,4,29488,apple_pay,393.66,2025-01-04 18:36:59,captured
4,5,63166,paypal,193.32,2026-01-19 07:12:21,refunded


### `shipments`

In [8]:
q("SELECT * FROM shipments LIMIT 5")

,shipment_id,order_id,carrier,tracking_number,shipped_at,delivered_at,status
0,1,63413,USPS,SW584565458EP,2026-08-01 00:00:00,2026-08-05 00:00:00,delivered
1,2,4440,FedEx,DQ236563249KO,2024-11-17 00:00:00,2024-11-23 00:00:00,delivered
2,3,58595,UPS,ZK657977356HP,2025-07-05 00:00:00,2025-07-12 00:00:00,delivered
3,4,620,FedEx,OQ057331203OQ,2024-02-17 00:00:00,2024-02-22 00:00:00,delivered
4,5,60606,FedEx,QY624001538CA,2026-01-23 00:00:00,2026-01-28 00:00:00,delivered


### `returns`

In [9]:
q("SELECT * FROM returns LIMIT 5")

,return_id,order_item_id,reason,refund_amount,requested_at,status
0,1,4766,wrong_size,16.31,2026-03-27 00:00:00,approved
1,2,102028,arrived_late,135.48,2026-05-15 00:00:00,refunded
2,3,190299,wrong_size,35.84,2026-04-24 00:00:00,refunded
3,4,131877,changed_mind,59.94,2024-12-23 00:00:00,refunded
4,5,193714,wrong_size,161.07,2024-12-13 00:00:00,refunded


## Ten questions

### 1. Monthly revenue and order count

In [10]:
q("""
SELECT substr(order_date, 1, 7) AS month, COUNT(*) AS orders, ROUND(SUM(total_amount), 2) AS revenue
FROM orders WHERE status <> 'cancelled'
GROUP BY month ORDER BY month;
""")

,month,orders,revenue
0,2024-01,1956,1130956.37
1,2024-02,1774,971562.31
2,2024-03,1880,1114181.45
3,2024-04,1857,1094484.01
4,2024-05,1934,1061563.61
5,2024-06,1804,1050848.03
6,2024-07,1889,1085871.53
7,2024-08,1910,1086564.02
8,2024-09,1846,1074365.95
9,2024-10,1903,1053467.82


### 2. Top 10 customers by lifetime spend

In [11]:
q("""
SELECT c.customer_id, c.first_name || ' ' || c.last_name AS customer, c.country,
       COUNT(*) AS orders, ROUND(SUM(o.total_amount), 2) AS spend
FROM orders o JOIN customers c USING (customer_id)
WHERE o.status <> 'cancelled'
GROUP BY c.customer_id ORDER BY spend DESC LIMIT 10;
""")

,customer_id,customer,country,orders,spend
0,15091,Barbara White,Canada,118,84701.61
1,20141,Anne Schultz,Canada,71,43579.69
2,24489,Monique Mendez,Spain,78,42025.98
3,5567,Jeanette Buchanan,France,67,34801.35
4,1560,Vanessa Chase,New Zealand,52,32859.13
5,15347,Aaron Graham,United States,50,31896.15
6,14489,Kelly Baxter,United States,58,29260.27
7,4268,Brandi Page,Canada,62,28761.35
8,15315,Christy Rubio,Germany,38,26264.62
9,13926,Jennifer Mayo,France,39,26108.94


### 3. Revenue and gross margin by category

In [12]:
q("""
SELECT p.category, ROUND(SUM(i.line_total), 2) AS revenue,
       ROUND(100.0 * SUM(i.line_total - i.quantity * p.unit_cost) / SUM(i.line_total), 1) AS margin_pct
FROM order_items i JOIN products p USING (product_id) JOIN orders o USING (order_id)
WHERE o.status NOT IN ('cancelled', 'returned')
GROUP BY p.category ORDER BY revenue DESC;
""")

,category,revenue,margin_pct
0,Tents,7944436.18,46.8
1,Backpacks,7059763.96,49.4
2,Navigation,6460394.98,49.3
3,Sleeping Bags,5133517.08,48.5
4,Footwear,4216056.49,42.2
5,Climbing,3429543.09,48.2
6,Apparel,2984647.05,49.1
7,Accessories,1974613.72,47.4
8,Cooking,1949003.11,46.8
9,Lighting,930502.86,48.4


### 4. How many customers order once, twice, three times or more?

In [13]:
q("""
SELECT CASE WHEN n = 1 THEN '1 order' WHEN n = 2 THEN '2 orders' WHEN n <= 5 THEN '3-5 orders' ELSE '6+ orders' END AS bucket,
       COUNT(*) AS customers
FROM (SELECT customer_id, COUNT(*) AS n FROM orders GROUP BY customer_id)
GROUP BY bucket ORDER BY MIN(n);
""")

,bucket,customers
0,1 order,3717
1,2 orders,6572
2,3-5 orders,12592
3,6+ orders,3119


### 5. Return rate by category

In [14]:
q("""
SELECT p.category, COUNT(r.return_id) AS returned_lines, COUNT(*) AS lines,
       ROUND(100.0 * COUNT(r.return_id) / COUNT(*), 2) AS return_rate_pct
FROM order_items i JOIN products p USING (product_id) LEFT JOIN returns r USING (order_item_id)
GROUP BY p.category ORDER BY return_rate_pct DESC;
""")

,category,returned_lines,lines,return_rate_pct
0,Lighting,542,12434,4.36
1,Climbing,782,18852,4.15
2,Navigation,546,13364,4.09
3,Backpacks,1368,33903,4.04
4,Footwear,750,18726,4.01
5,Cooking,1011,25728,3.93
6,Apparel,832,21160,3.93
7,Accessories,1082,27515,3.93
8,Sleeping Bags,513,13164,3.90
9,Tents,574,15154,3.79


### 6. Average delivery time by carrier, in days

In [15]:
q("""
SELECT carrier, COUNT(*) AS deliveries,
       ROUND(AVG(julianday(delivered_at) - julianday(shipped_at)), 2) AS avg_days
FROM shipments WHERE delivered_at IS NOT NULL
GROUP BY carrier ORDER BY avg_days;
""")

,carrier,deliveries,avg_days
0,Canada Post,3411,5.42
1,DHL,14704,5.48
2,DPD,4958,5.50
3,USPS,13491,5.50
4,Royal Mail,6729,5.51
5,FedEx,18222,5.52
6,UPS,21656,5.52


### 7. Average order value by channel and loyalty tier

In [16]:
q("""
SELECT o.channel, c.loyalty_tier, COUNT(*) AS orders, ROUND(AVG(o.total_amount), 2) AS avg_order_value
FROM orders o JOIN customers c USING (customer_id)
GROUP BY o.channel, c.loyalty_tier ORDER BY o.channel, avg_order_value DESC;
""")

,channel,loyalty_tier,orders,avg_order_value
0,app,platinum,980,578.39
1,app,none,21411,568.66
2,app,silver,5260,565.98
3,app,gold,2995,565.75
4,marketplace,none,8926,568.48
5,marketplace,silver,2204,564.58
6,marketplace,platinum,392,556.15
7,marketplace,gold,1225,540.29
8,store,silver,938,633.24
9,store,platinum,151,619.71


### 8. Share of orders that got free shipping, per year

In [17]:
q("""
SELECT substr(order_date, 1, 4) AS year,
       ROUND(100.0 * SUM(shipping_fee = 0) / COUNT(*), 1) AS free_shipping_pct
FROM orders GROUP BY year;
""")

,year,free_shipping_pct
0,2024,91.4
1,2025,91.4
2,2026,91.4


### 9. Products most often bought together

In [18]:
q("""
SELECT a.product_id AS product_a, b.product_id AS product_b, COUNT(*) AS orders_together
FROM order_items a JOIN order_items b ON a.order_id = b.order_id AND a.product_id < b.product_id
GROUP BY a.product_id, b.product_id ORDER BY orders_together DESC LIMIT 10;
""")

,product_a,product_b,orders_together
0,490,518,700
1,518,722,504
2,518,1154,412
3,324,518,333
4,490,722,304
5,518,694,282
6,490,1154,235
7,167,518,223
8,518,1167,207
9,193,518,204


### 10. Running total of revenue per month (window function)

In [19]:
q("""
SELECT month, revenue, ROUND(SUM(revenue) OVER (ORDER BY month), 2) AS running_revenue
FROM (SELECT substr(order_date, 1, 7) AS month, ROUND(SUM(total_amount), 2) AS revenue
      FROM orders WHERE status <> 'cancelled' GROUP BY month)
ORDER BY month;
""")

,month,revenue,running_revenue
0,2024-01,1130956.37,1130956.37
1,2024-02,971562.31,2102518.68
2,2024-03,1114181.45,3216700.13
3,2024-04,1094484.01,4311184.14
4,2024-05,1061563.61,5372747.75
5,2024-06,1050848.03,6423595.78
6,2024-07,1085871.53,7509467.31
7,2024-08,1086564.02,8596031.33
8,2024-09,1074365.95,9670397.28
9,2024-10,1053467.82,10723865.10


## Where to go next

- Change a query above: another grouping, another filter, a different window.
- The dataset's rules (keys match, totals add up, dates are in order) are written as queries in [checks.sql](https://github.com/generaterealdata/synthetic-datasets/blob/main/ecommerce/checks.sql). Each one returns 0.
- The data is fully synthetic, generated with [GenerateRealData](https://generaterealdata.com/use-cases/relational-database-test-data/). The schema, the generation plan and a data card with every column explained are on [GitHub](https://github.com/generaterealdata/synthetic-datasets/tree/main/ecommerce).